# Truthprint B2 — 다국어 의미기반 실측 (Kaggle / Colab / Local)

**사람 주석 불필요.** 소스 생성 → NLLB로 5개 언어(+round-trip) 번역 → eval 실행 → zip.
셀을 위에서부터 실행하세요.

> ⚠️ **Kaggle:** 오른쪽 Settings → **Internet: On** (모델 다운로드 필요). Accelerator는
> None(CPU)로 충분하며, GPU가 있으면 번역이 더 빠릅니다.

## 셀 1 — 환경 감지 + 클론 + 설치

In [ ]:
import os, sys, subprocess
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else (
        '/content' if os.path.isdir('/content') else os.getcwd())
REPO = os.path.join(BASE, 'truthprint')
if not os.path.isdir(REPO):
    r = subprocess.run(['git','clone','--depth','1',
                        'https://github.com/leemgs/truthprint', REPO])
    if r.returncode != 0:
        raise RuntimeError('git clone 실패 -> 인터넷(Internet: On)을 확인하세요.')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',
                os.path.join(REPO,'code')])
sys.path.insert(0, os.path.join(REPO,'code'))
WORK = os.path.join(BASE, 'b2_data'); os.makedirs(WORK, exist_ok=True)
SCRIPTS = os.path.join(REPO,'code','scripts')
print('BASE =', BASE); print('WORK =', WORK); print('setup OK')

## 셀 2 — 실제-규모 영어 소스 생성 (문서 수 조절 가능)

In [ ]:
import json, random
from truthprint import challenge as ch
N_DOCS, SENTS = 10, 16     # 문서↑ = 신뢰구간↑(시간↑). 첫 실행은 10 권장, 5로 파일럿도 가능.
rng = random.Random(20260924); rows = []
for d in range(N_DOCS):
    doc = f'D{d+1:04d}'; facts = [ch.sample_fact(rng) for _ in range(SENTS)]
    wm, fr = {}, []
    for i, f in enumerate(facts):
        sid = f'{doc}-s{i+1}'; wm[sid] = ch.realize(f, 0, 0)
        fr.append(dict(ch.ext_invariants(f), sent_id=sid))
    rows.append({'doc_id': doc, 'lang_src': 'en', 'facts': fr, 'watermarked_text': wm})
with open(os.path.join(WORK,'01_source_items.jsonl'),'w',encoding='utf-8') as fh:
    for r in rows: fh.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'generated {N_DOCS} docs x {SENTS} sents')

## 셀 3 — NLLB로 5개 언어 + round-trip 번역 (실제 MT)
모델 최초 다운로드는 약 2.4GB(1회). 문서 수에 비례해 시간이 늘어납니다.

In [ ]:
import subprocess, sys, json
subprocess.run([sys.executable,'-m','pip','install','-q','transformers','sentencepiece','torch'])
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MODEL = 'facebook/nllb-200-distilled-600M'; SYSTEM = f'HuggingFace/{MODEL}'
TGT = ['ko','hi','zh','ar','de']    # 대상 언어(추가/삭제 가능)
NLLB = {'en':'eng_Latn','ko':'kor_Hang','hi':'hin_Deva',
        'zh':'zho_Hans','ar':'arb_Arab','de':'deu_Latn'}
print('loading model (~2.4GB, first time only)...')
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
def tr(t, s, g):
    tok.src_lang = NLLB[s]; enc = tok(t, return_tensors='pt')
    bos = tok.convert_tokens_to_ids(NLLB[g])
    o = model.generate(**enc, forced_bos_token_id=bos, max_length=256)
    return tok.batch_decode(o, skip_special_tokens=True)[0]
rows = [json.loads(l) for l in open(os.path.join(WORK,'01_source_items.jsonl'),encoding='utf-8')]
tf = []
for r in rows:
    for sid, text in r['watermarked_text'].items():
        for lg in TGT:
            tf.append({'transform_id':f'{sid}-{lg}','doc_id':r['doc_id'],'sent_id':sid,
                       'transform_type':'translation','direction':f'en->{lg}','system':SYSTEM,
                       'params':{},'output_text':tr(text,'en',lg),'round_trip':False})
        rt = tr(tr(text,'en','ko'),'ko','en')
        tf.append({'transform_id':f'{sid}-rt','doc_id':r['doc_id'],'sent_id':sid,
                   'transform_type':'roundtrip_translation','direction':'en->ko->en','system':SYSTEM,
                   'params':{},'output_text':rt,'round_trip':True})
    print('translated', r['doc_id'])
with open(os.path.join(WORK,'02_transformations.jsonl'),'w',encoding='utf-8') as fh:
    for x in tf: fh.write(json.dumps(x, ensure_ascii=False) + '\n')
print('wrote', len(tf), 'translations')

## 셀 4 — 실측 실행 (필드 복원율 + 의미기반 provenance 인증)
두 표가 이 데이터의 정직한 실측입니다(ko/hi/zh/ar/de/round-trip).

In [ ]:
import subprocess, sys, os
for name in ['eval_multilingual.py','eval_provenance.py']:
    print('='*72); print('RUN', name); print('='*72)
    out = os.path.join(WORK, name.replace('eval_','').replace('.py','')+'.md')
    r = subprocess.run([sys.executable, os.path.join(SCRIPTS,name), WORK, '--out', out],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)

## 셀 5 — 결과 zip 만들기 → 나에게 전달

In [ ]:
import shutil, os
zip_path = shutil.make_archive(os.path.join(BASE,'b2_results'), 'zip', WORK)
print('created:', zip_path)
downloaded = False
try:
    from google.colab import files; files.download(zip_path); downloaded = True
except Exception:
    pass
if not downloaded:
    try:
        from IPython.display import FileLink, display
        print('아래 링크 클릭해 다운로드 (Kaggle/로컬):'); display(FileLink(os.path.relpath(zip_path, os.getcwd())))
    except Exception:
        print('수동 다운로드 경로:', zip_path)
    print('Kaggle이면 오른쪽 파일 패널에서 b2_results.zip 우클릭 -> Download 도 가능합니다.')